# Alchemy Effects Data Pipeline

This notebook documents the full pipeline for scraping, cleaning, and transforming alchemy reagents and their effects from the [Unofficial Elder Scrolls Pages (UESP)](https://en.uesp.net/wiki/Lore:Alchemy).


## 1. Project Overview

The project scrapes alchemy reagents and their effects from [UESP alchemy wiki pages](https://en.uesp.net/wiki/Lore:Alchemy) and processes the collected data through a series of standardization and cleaning steps before exporting the final result as a text file.


**Pipeline steps:**
1. `script1_scrape.py` - Scrapes reagent names and effects from the website pages into `01_parsed.txt`
2. `script2_rename.py` - Loads `01_parsed.txt` and replaces effect names with pre-designed standardized names into `02_renamed.txt`
3. `script3_clean.py` - Loads `02_renamed.txt` and performs final cleaning and deduplicating into `03_cleaned_(final).txt`

**Note on Execution Context:**

The original scripts write results directly to files.  
In this notebook, file I/O has been replaced with in-memory structures and printed output for immediate demonstration.

## 2. Folder Structure

```
project_root/
│ main.py                  # Runs the pipeline
│ README.md
│ data/                    # Stores intermediate and final text files
│ scripts/
│   ├ script1_scrape.py
│   ├ script2_rename.py
│   └ script3_clean.py
│ notebooks/
│   └ alchemy_scraping_pipeline.ipynb
```

## 3. Pipeline Overview

In [ ]:
Raw TES Sources
        ↓
Web Scraping
        ↓
01_parsed.txt
        ↓
Effect Renaming
        ↓
02_renamed.txt
        ↓
Cleaning & Deduplication
        ↓
03_cleaned_(final).txt

### Step 1 — Scraping Reagents and Effects

**Script:** `script1_scrape.py`

- Scrapes all reagents and their effects from the UESP Alchemy pages (A–Z).
- Appends NAME to reagent lines to distinguish reagent lines from effect lines.
- Effects lines are indented for clarity.

In [1]:
# Retrieve data from UESP Alchemy pages (A–Z) with Requests and BeautifulSoup:

import re
import time

import requests
from bs4 import BeautifulSoup as bs

# Source URLs:
urls = [
    "https://en.uesp.net/wiki/Lore:Alchemy_A",
]

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"
    }

skip_headings = {"References", "See also", "External links"}  # Non-reagent headings to ignore
effect_classes = {"EffectNeg", "EffectPos", "EffectOther", "EffectMix"}  # Target classes to extract effects

for url in urls:
    r = requests.get(url, headers=headers)

    soup = bs(r.content, "html.parser")
    tags = soup.find_all(class_=re.compile("mw-headline|Effect(Neg|Pos|Other|Mix)"))  # Select headings and effect elements
    

    for tag in tags:


        # Parse reagents and append NAME to reagent lines for distinguishability:

        if "mw-headline" in tag["class"]:  # Filter reagent name headings
            reagent = tag.get_text().strip()
            if reagent in skip_headings:
                continue
            print(reagent + " NAME")
            seen_effects = set()  # Track seen effects to avoid duplicates
        
        
        # Parse effects and add indentation for each effect line:
        
        elif any(
            c in effect_classes for c in tag["class"]  # Identify effect elements
        ):
            effects = tag.get_text(separator="\n").strip()  # Preserve multiple effects by separating text nodes
            for effect in effects.split("\n"):  # Split into individual effects
                effect = effect.strip()
                if effect in seen_effects:
                    continue
                else:
                    seen_effects.add(effect)
                    print("    " + effect)


    time.sleep(1.2)  # Rate limiting between requests (respectful scraping practice)

Abecean Longfin NAME
    Weakness to Frost
    Fortify Sneak
    Weakness to Poison
    Fortify Restoration
Adamantium Ore NAME
    Burden
    Restore Magicka
    Poison
    Reflect
Ahm Galath NAME
    Increase Lethal Strike
Alit Hide NAME
    Drain Intelligence
    Resist Poison
    Telekinesis
    Detect Animal
Alkanet Flower NAME
    Restore Intelligence
    Resist Poison
    Light
    Damage Fatigue
Alocasia Fruit NAME
    Restore Fatigue
    Light
    Restore Health
    Damage Magicka
    Regenerate Stamina
    Ravage Magicka
    Regenerate Health
Aloe Vera Leaves NAME
    Heal Health
    Restore Health
    Restore Fatigue
    Damage Magicka
    Invisibility
    Restore Stamina
Amber NAME
    Resist Fire
Ambrosia NAME
    Restore Health
    Regenerate Health
    Fortify Health
    Cure Poison
Ampoule Pod NAME
    Water Walking
    Paralyze
    Detect Animal
    Drain Willpower
Ancestor Moth Wing NAME
    Damage Stamina
    Fortify Conjuration
    Damage Magicka Regen
    Fortify E

<details>
<summary><strong>Expected Output</strong></summary>

```
Abecean Longfin NAME
    Weakness to Frost
    Fortify Sneak
    Weakness to Poison
    Fortify Restoration
Adamantium Ore NAME
    Burden
    Restore Magicka
    Poison
    Reflect
Ahm Galath NAME
    Increase Lethal Strike
Alit Hide NAME
    Drain Intelligence
    Resist Poison
    Telekinesis
    Detect Animal
Alkanet Flower NAME
    Restore Intelligence
    Resist Poison
    Light
    Damage Fatigue
Alocasia Fruit NAME
    Restore Fatigue
    Light
    Restore Health
    Damage Magicka
    Regenerate Stamina
    Ravage Magicka
    Regenerate Health
Aloe Vera Leaves NAME
    Heal Health
    Restore Health
    Restore Fatigue
    Damage Magicka
    Invisibility
    Restore Stamina
Amber NAME
    Resist Fire
Ambrosia NAME
    Restore Health
    Regenerate Health
    Fortify Health
    Cure Poison
Ampoule Pod NAME
    Water Walking
    Paralyze
    Detect Animal
    Drain Willpower
Ancestor Moth Wing NAME
    Damage Stamina
    Fortify Conjuration
    Damage Magicka Regen
    Fortify Enchanting
Angelfish NAME
    Regenerate Health
    Resist Fire
    Fortify Marksman
    Waterbreathing
Angler Larva NAME
    Lingering Damage Health
    Regenerate Stamina
    Waterbreathing
    Fortify Two-handed
Antlers NAME
    Restore Stamina
    Fortify Stamina
    Slow
    Damage Stamina Regen
    Weakness to Poison
    Fortify Restoration
    Lingering Damage Stamina
    Damage Health
Apple NAME
    Restore Fatigue
    Damage Luck
    Fortify Willpower
    Damage Health
    Restore Health
Arrowroot NAME
    Restore Agility
    Damage Luck
    Fortify Strength
    Burden
Ash Creep Cluster NAME
    Damage Stamina
    Invisibility
    Resist Fire
    Fortify Destruction
Ash Hopper Jelly NAME
    Restore Health
    Fortify Light Armor
    Resist Shock
    Weakness to Frost
Ash Salt NAME
    Drain Agility
    Resist Magicka
    Cure Blight Disease
Ash Yam NAME
    Fortify Intelligence
    Fortify Strength
    Resist Common Disease
    Detect Key
    Restore Health
Ashen Grass Pod NAME
    Resist Fire
    Weakness to Shock
    Fortify Lockpicking
    Fortify Sneak
Ashen Remains NAME
    Fortify Fatigue
    Damage Luck
    Silence
    Weakness to Fire
Aster Bloom Core NAME
    Restore Agility
    Dispel
    Shield
    Burden
    Resist Magic
    Fortify Light Armor
    Fortify Block
    Paralysis
Azra Root NAME
    Add Intelligence
    Damage Endurance
    Damage Strength
```

### Step 2 — Renaming Effects

**Script:** `script2_rename.py`

- Uses a predefined mapping dictionary to standardize effect names according to the new naming scheme.

In [ ]:
# Retrieved data sample:

scraped_regents = '''
Coda Flower NAME
    Drain Personality
    Levitate
    Drain Intelligence
    Drain Health
    Damage Health
    Lingering Damage Stamina
    Ravage Magicka
    Fortify Carry Weight
Columbine NAME
    Restore Personality
    Resist Frost
    Fortify Magicka
    Chameleon
    Restore Health
    Restore Magicka
    Restore Stamina
    Unstoppable
'''


# Mapping dictionary sample:

replacements = {
    "Restore Health": "Health Restoration",
    "Damage Health": "Health Damage",
    "Drain Health": "Health Damage",
    "Ravage Magicka": "Magicka Damage",
    "Restore Magicka": "Magicka Restoration",
    "Fortify Magicka": "Increased Magicka",
    "Restore Stamina": "Stamina Restoration",
    "Lingering Damage Stamina": "Lingering Stamina Damage",
    "Drain Intelligence": "Intelligence Decrease",
    "Fortify Carry Weight": "Strength Increase",
    "Restore Personality": "Personality Restoration",
    "Drain Personality": "Personality Decrease",
    "Resist Frost": "Frost Resistance",
    "Unstoppable": "Paralysis Resistance",
    "Levitate": "Levitation",
    "Chameleon": "Chameleon",
}


# Rename:

for line in scraped_regents.split('\n'):
       line = line.strip()
       if line in replacements:
              line = replacements[line]
              line = "\t" + line
       print(line)


Coda Flower NAME
	Personality Decrease
	Levitation
	Intelligence Decrease
	Health Damage
	Health Damage
	Lingering Stamina Damage
	Magicka Damage
	Strength Increase
Columbine NAME
	Personality Restoration
	Frost Resistance
	Increased Magicka
	Chameleon
	Health Restoration
	Magicka Restoration
	Stamina Restoration
	Paralysis Resistance



<details>
<summary><strong>Expected Output</strong></summary>

```
Coda Flower NAME
	Personality Decrease
	Levitation
	Intelligence Decrease
	Health Damage
	Health Damage
	Lingering Stamina Damage
	Magicka Damage
	Strength Increase
Columbine NAME
	Personality Restoration
	Frost Resistance
	Increased Magicka
	Chameleon
	Health Restoration
	Magicka Restoration
	Stamina Restoration
	Paralysis Resistance
```

### Step 3 — Cleaning and Deduplicating

**Script:** `script3_clean.py`

- Removes the NAME marker from reagent lines.
- Ensures no effect is repeated for a reagent (deduplicating).
- Additional minor formatting.

**Code Snippet:**

In [2]:
# Renamed data sample:

renamed_effects = '''
Coda Flower NAME
	Personality Decrease
	Levitation
	Intelligence Decrease
	Health Damage
	Health Damage
	Lingering Stamina Damage
	Magicka Damage
	Strength Increase
Columbine NAME
	Personality Restoration
	Frost Resistance
	Increased Magicka
	Chameleon
	Health Restoration
	Magicka Restoration
	Stamina Restoration
	Paralysis Resistance
'''


# Cleaning / deduplicating function:

def clean_effects(data):
    effects = {}

    current_reagent = None
    for line in data.split('\n'):
        if "NAME" in line:  # Identify reagent line
            current_reagent = line.strip()[:-5]  # Remove trailing "NAME" suffix
            effects[current_reagent] = []  # Initialize list for current reagent's effects
        else:
            effect = line.strip()
            if not effect:
                continue
            if effect not in effects[current_reagent]:  # Prevent duplicate effects
                effects[current_reagent].append(effect)
    return effects  # The result is {reagent:[effects list]} dictionary


# Call function:

clean_data = clean_effects(renamed_effects)


# Parse through function's result for final formatted delivery:

for reagent, effect_list in clean_data.items():
    print(reagent)
    for effect in effect_list:
        print(f"    {effect}")
    print()  # Separate reagent blocks with new line

Coda Flower
    Personality Decrease
    Levitation
    Intelligence Decrease
    Health Damage
    Lingering Stamina Damage
    Magicka Damage
    Strength Increase

Columbine
    Personality Restoration
    Frost Resistance
    Increased Magicka
    Chameleon
    Health Restoration
    Magicka Restoration
    Stamina Restoration
    Paralysis Resistance



<details>
<summary><strong>Expected Output</strong></summary>

```
Coda Flower
    Personality Decrease
    Levitation
    Intelligence Decrease
    Health Damage
    Lingering Stamina Damage
    Magicka Damage
    Strength Increase

Columbine
    Personality Restoration
    Frost Resistance
    Increased Magicka
    Chameleon
    Health Restoration
    Magicka Restoration
    Stamina Restoration
    Paralysis Resistance
```

## 4. Running the Full Pipeline

All scripts can be run sequentially from `main.py` using subprocess.

In [ ]:
import subprocess

subprocess.run(["python", "scripts/script1_scrape.py"])
subprocess.run(["python", "scripts/script2_rename.py"])
subprocess.run(["python", "scripts/script3_clean.py"])

## 5. Notes

- Ensure `data/` folder exists before running the scripts.
- Intermediate files are stored in `data/`.
- Check `03_cleaned_(final).txt` for the final output.
- Update the replacement dictionary if new effect names are introduced.
- Update `skip_headings` if UESP introduces additional non-reagent sections.